In [15]:
import os
from itertools import product
import shutil
# Parameters
atwood_list = [0.5]#, 0.5]
reynolds_list = [300, 1000]
surface_tension_list = [1e-5]#, 1e-4, 5e-4]
delta_h = [0.0045]

def generate_ini(path, delta_h, amplitude, rho_high, rho_low, sigma, time_end, timestep, timestep_freq, viscosity):
    ini_template = """-param delta_h {delta_h}
-param amplitude {amplitude}
-param high_rho {rho_high}
-param low_rho {rho_low}
-param surface_tension {surface_tension}
-param time_end {time_end}
-param timestep {timestep}
-param timestep_freq {timestep_freq}
-param viscosity_high {viscosity} 
-param viscosity_low {viscosity}
"""

#-param high_visc {viscosity} 
#-param low_visc {viscosity}
#"""
#viscosity_high
    ini_contents = ini_template.format(
        delta_h=delta_h,
        amplitude=amplitude,
        rho_high=rho_high,
        rho_low=rho_low,
        surface_tension=sigma,
        time_end=time_end,
        timestep=timestep,
        timestep_freq = timestep_freq,
        viscosity=viscosity
    )
    with open(os.path.join(path, "params.ini"), "w") as f:
        f.write(ini_contents)
    return ini_contents

def generate_slurm(A, Re, sigma, path):
    slurm_script = f"""#!/usr/bin/zsh

#SBATCH --job-name=A{A}_Re{Re}_sig{sigma:.0e}
#SBATCH --output=A{A}_Re{Re}_sig{sigma:.0e}_%J.out
#SBATCH --time=08:30:00
#SBATCH --ntasks-per-node=4
#SBATCH --nodes=2

cd {os.path.abspath(path)}
module load STAR-CCM+/18.06.006

export CDLMD_LICENSE_FILE=50015@license.itc.rwth-aachen.de

starccm+ -ini params.ini -bs slurm -mpi intel -xsystemucx -rsh ssh -power -batch mesh,run 0main_surf.sim
"""
    with open(os.path.join(path, "run_sim.sh"), "w") as f:
        f.write(slurm_script)   

In [16]:
base_dir = "/home/yy310050/Desktop/thesis/rayleigh_taylor/final_sims/Re3000_Amp01"

In [17]:
for A, Re, sigma, delta_h in product(atwood_list, reynolds_list, surface_tension_list, delta_h):
    rho_low = 1
    time_end = 6.5
    amplitude = 0.1
    folder_name = f"Re{Re}_At{A}_sigma{sigma:.0e}_dh{delta_h}"
    path = os.path.join(base_dir, folder_name)
    os.makedirs(path, exist_ok=True)
    print(path)

    if A == 0.5:
        rho_high = 3
        timestep = 0.0035
        timestep_freq = '1/(15*${timestep})' #ca 19
        #timestep = 1.75e-3#0.0035 
        #timestep_freq = '1/(30*${timestep})' 
        if Re == 300:
            viscosity = 0.01
        elif Re == 1000:
            viscosity = 0.003
        elif Re == 3000:
            viscosity = 0.001
    
    elif A == 0.2:
        rho_high = 1.5
        timestep = 0.0055
        timestep_freq = '1/(10*${timestep})'
        if Re == 300:
            viscosity = 0.005
        elif Re == 1000:
            viscosity = 0.0015
        elif Re == 3000:
            viscosity = 0.0005
    
    
    ini_contents = generate_ini(path, delta_h, amplitude, rho_high, rho_low, sigma, time_end, timestep, timestep_freq, viscosity)
    generate_slurm(A, Re, sigma, path)


    
    shutil.copy("0main_surf.sim", path)
    print('\n\n\n')

/home/yy310050/Desktop/thesis/rayleigh_taylor/final_sims/Re3000_Amp01/Re300_At0.5_sigma1e-05_dh0.0045




/home/yy310050/Desktop/thesis/rayleigh_taylor/final_sims/Re3000_Amp01/Re1000_At0.5_sigma1e-05_dh0.0045




